Part 1 — DataFrame (PySpark)


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, sum, avg, count, round, desc

spark = SparkSession.builder.appName("HospitalAssessment").getOrCreate()

data = [
    (101, "Arjun Reddy",  "Hyderabad",  "Cardiology",   5000, 1),
    (102, "Sneha Kapoor", "Delhi",       "Orthopedics",  3000, 2),
    (103, "Rahul Sharma", "Mumbai",      "Dermatology",  1500, 1),
    (104, "Priya Nair",   "Bangalore",   "Cardiology",   5000, 2),
    (105, "Vikram Singh", "Chennai",     "Neurology",    7000, 1),
    (106, "Ananya Das",   "Kolkata",     "Orthopedics",  3000, 3),
    (107, "Karan Patel",  "Ahmedabad",   "Cardiology",   5000, 1),
    (108, "Meera Iyer",   "Bangalore",   "Dermatology",  1500, 2)
]
columns = ["visit_id", "patient_name", "city", "department", "consultation_fee", "tests_count"]

df = spark.createDataFrame(data, columns)
df.show()
df.printSchema()


+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+

root
 |-- visit_id: long (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: s

In [0]:

df = df.withColumn("total_bill", col("consultation_fee") + (col("tests_count") * 500)) \
       .withColumn("patient_category",
           when(col("consultation_fee") >= 5000, "High")
          .when(col("consultation_fee") >= 3000, "Medium")
          .otherwise("Low"))
df.show()


+--------+------------+---------+-----------+----------------+-----------+----------+----------------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total_bill|patient_category|
+--------+------------+---------+-----------+----------------+-----------+----------+----------------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|      5500|            High|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|      4000|          Medium|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|      2000|             Low|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      6000|            High|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|      7500|            High|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|      4500|          Medium|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1

In [0]:
high_value_df = df.filter(col("consultation_fee") >= 5000)
high_value_df.show()


+--------+------------+---------+----------+----------------+-----------+----------+----------------+
|visit_id|patient_name|     city|department|consultation_fee|tests_count|total_bill|patient_category|
+--------+------------+---------+----------+----------------+-----------+----------+----------------+
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|          1|      5500|            High|
|     104|  Priya Nair|Bangalore|Cardiology|            5000|          2|      6000|            High|
|     105|Vikram Singh|  Chennai| Neurology|            7000|          1|      7500|            High|
|     107| Karan Patel|Ahmedabad|Cardiology|            5000|          1|      5500|            High|
+--------+------------+---------+----------+----------------+-----------+----------+----------------+



In [0]:
dept_agg = df.groupBy("department").agg(
    count("visit_id").alias("patient_count"),
    sum("consultation_fee").alias("total_revenue"),
    round(avg("consultation_fee"), 2).alias("avg_fee")
)
dept_agg.show()


+-----------+-------------+-------------+-------+
| department|patient_count|total_revenue|avg_fee|
+-----------+-------------+-------------+-------+
| Cardiology|            3|        15000| 5000.0|
|Orthopedics|            2|         6000| 3000.0|
|Dermatology|            2|         3000| 1500.0|
|  Neurology|            1|         7000| 7000.0|
+-----------+-------------+-------------+-------+



In [0]:
sorted_df = dept_agg.orderBy(desc("total_revenue"))
sorted_df.show()


+-----------+-------------+-------------+-------+
| department|patient_count|total_revenue|avg_fee|
+-----------+-------------+-------------+-------+
| Cardiology|            3|        15000| 5000.0|
|  Neurology|            1|         7000| 7000.0|
|Orthopedics|            2|         6000| 3000.0|
|Dermatology|            2|         3000| 1500.0|
+-----------+-------------+-------------+-------+



Part 2 — Spark SQL


In [0]:
df.createOrReplaceTempView("hospital_visits")

In [0]:
spark.sql("""
SELECT * FROM hospital_visits
WHERE department = 'Cardiology'
""").show()


+--------+------------+---------+----------+----------------+-----------+----------+----------------+
|visit_id|patient_name|     city|department|consultation_fee|tests_count|total_bill|patient_category|
+--------+------------+---------+----------+----------------+-----------+----------+----------------+
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|          1|      5500|            High|
|     104|  Priya Nair|Bangalore|Cardiology|            5000|          2|      6000|            High|
|     107| Karan Patel|Ahmedabad|Cardiology|            5000|          1|      5500|            High|
+--------+------------+---------+----------+----------------+-----------+----------+----------------+



In [0]:
spark.sql("""
SELECT city,
       SUM(consultation_fee) AS total_revenue,
       COUNT(visit_id)        AS patient_count
FROM hospital_visits
GROUP BY city
ORDER BY total_revenue DESC
""").show()


+---------+-------------+-------------+
|     city|total_revenue|patient_count|
+---------+-------------+-------------+
|  Chennai|         7000|            1|
|Bangalore|         6500|            2|
|Hyderabad|         5000|            1|
|Ahmedabad|         5000|            1|
|    Delhi|         3000|            1|
|  Kolkata|         3000|            1|
|   Mumbai|         1500|            1|
+---------+-------------+-------------+



In [0]:
spark.sql("""
SELECT visit_id,
       patient_name,
       department,
       consultation_fee,
       (consultation_fee + tests_count * 500) AS total_bill
FROM hospital_visits
ORDER BY total_bill DESC
LIMIT 3
""").show()


+--------+------------+----------+----------------+----------+
|visit_id|patient_name|department|consultation_fee|total_bill|
+--------+------------+----------+----------------+----------+
|     105|Vikram Singh| Neurology|            7000|      7500|
|     104|  Priya Nair|Cardiology|            5000|      6000|
|     101| Arjun Reddy|Cardiology|            5000|      5500|
+--------+------------+----------+----------------+----------+



In [0]:
spark.sql("""
SELECT department, COUNT(*) AS patient_count
FROM hospital_visits
GROUP BY department
ORDER BY patient_count DESC
""").show()


+-----------+-------------+
| department|patient_count|
+-----------+-------------+
| Cardiology|            3|
|Orthopedics|            2|
|Dermatology|            2|
|  Neurology|            1|
+-----------+-------------+



Part 3 — Delta Lake 


In [0]:
spark.sql("DROP TABLE IF EXISTS hospital_visits")

from pyspark.sql import SparkSession
data = [
    (101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
    (102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
    (103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
    (104,"Priya Nair","Bangalore","Cardiology",5000,2),
    (105,"Vikram Singh","Chennai","Neurology",7000,1),
    (106,"Ananya Das","Kolkata","Orthopedics",3000,3),
    (107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
    (108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]
columns = ["visit_id","patient_name","city","department","consultation_fee","tests_count"]
base_df = spark.createDataFrame(data, columns)

base_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("hospital_visits")

spark.sql("SELECT * FROM hospital_visits").show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



In [0]:
spark.sql("""
INSERT INTO hospital_visits VALUES
  (109, 'Rohan Mehta',  'Pune',    'Neurology',  7000, 2),
  (110, 'Divya Thomas', 'Chennai', 'Cardiology', 5000, 1)
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
UPDATE hospital_visits
SET consultation_fee = 5500
WHERE department = 'Cardiology'
""")

DataFrame[num_affected_rows: bigint]

In [0]:
spark.sql("""
DELETE FROM hospital_visits
WHERE department = 'Dermatology'
""")

DataFrame[num_affected_rows: bigint]

In [0]:
new_data = [
    (103, "Rahul Sharma", "Mumbai", "Dermatology", 2000, 2),
    (111, "Sonal Gupta",  "Jaipur", "Orthopedics", 3500, 1)
]
columns = ["visit_id","patient_name","city","department","consultation_fee","tests_count"]
new_df = spark.createDataFrame(new_data, columns)
new_df.createOrReplaceTempView("new_hospital_visits")

spark.sql("""
MERGE INTO hospital_visits AS target
USING new_hospital_visits AS source
ON target.visit_id = source.visit_id
WHEN MATCHED THEN
  UPDATE SET
    target.consultation_fee = source.consultation_fee,
    target.tests_count      = source.tests_count
WHEN NOT MATCHED THEN
  INSERT *
""")

spark.sql("SELECT * FROM hospital_visits ORDER BY visit_id").show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5500|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            2000|          2|
|     104|  Priya Nair|Bangalore| Cardiology|            5500|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5500|          1|
|     109| Rohan Mehta|     Pune|  Neurology|            7000|          2|
|     110|Divya Thomas|  Chennai| Cardiology|            5500|          1|
|     111| Sonal Gupta|   Jaipur|Orthopedics|            3500|          1|
+--------+------------+--

Part 4 — Delta Advanced


In [0]:
spark.sql("DESCRIBE HISTORY hospital_visits").show(truncate=False)


+-------+-------------------+---------------+-------------------------------------------------------+---------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
spark.sql("""
SELECT * FROM hospital_visits VERSION AS OF 0
""").show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



In [0]:

print("VACUUM effect: Deletes stale data files, breaks time travel beyond retention window.")


VACUUM effect: Deletes stale data files, breaks time travel beyond retention window.


In [0]:
spark.sql("""
VACUUM hospital_visits RETAIN 168 HOURS DRY RUN
""").show(truncate=False)


+----+
|path|
+----+
+----+



Part 5 — Parquet → Delta


In [0]:
base_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("hospital_visits_parquet_source")

print("Source table saved (simulating Parquet layer)")
spark.sql("SELECT * FROM hospital_visits_parquet_source").show()

Source table saved (simulating Parquet layer)
+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



In [0]:
spark.sql("""
  CREATE OR REPLACE TABLE hospital_visits_delta_converted
  USING DELTA
  AS SELECT * FROM hospital_visits_parquet_source
""")

print("Converted to Delta table successfully")

Converted to Delta table successfully


In [0]:
spark.sql("DESCRIBE EXTENDED hospital_visits_delta_converted").show(truncate=False)
spark.sql("SELECT * FROM hospital_visits_delta_converted").show()

+----------------------------+-----------------------------------------------------------------------+-------+
|col_name                    |data_type                                                              |comment|
+----------------------------+-----------------------------------------------------------------------+-------+
|visit_id                    |bigint                                                                 |NULL   |
|patient_name                |string                                                                 |NULL   |
|city                        |string                                                                 |NULL   |
|department                  |string                                                                 |NULL   |
|consultation_fee            |bigint                                                                 |NULL   |
|tests_count                 |bigint                                                                 |NULL   |
|

Part 6 — Incremental Load


In [0]:
spark.sql("DROP TABLE IF EXISTS hospital_visits_target")

base_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("hospital_visits_target")

spark.sql("SELECT * FROM hospital_visits_target ORDER BY visit_id").show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



In [0]:
daily_updates = [
    (104, "Priya Nair",   "Bangalore", "Cardiology",  6000, 3),
    (112, "Nikhil Joshi", "Nagpur",    "Neurology",   7000, 2),
    (113, "Pooja Sharma", "Surat",     "Orthopedics", 3200, 1)
]
columns = ["visit_id","patient_name","city","department","consultation_fee","tests_count"]
updates_df = spark.createDataFrame(daily_updates, columns)
updates_df.createOrReplaceTempView("daily_updates")
updates_df.show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     104|  Priya Nair|Bangalore| Cardiology|            6000|          3|
|     112|Nikhil Joshi|   Nagpur|  Neurology|            7000|          2|
|     113|Pooja Sharma|    Surat|Orthopedics|            3200|          1|
+--------+------------+---------+-----------+----------------+-----------+



In [0]:
spark.sql("""
MERGE INTO hospital_visits_target AS target
USING daily_updates AS source
ON target.visit_id = source.visit_id
WHEN MATCHED THEN
  UPDATE SET
    target.patient_name     = source.patient_name,
    target.city             = source.city,
    target.department       = source.department,
    target.consultation_fee = source.consultation_fee,
    target.tests_count      = source.tests_count
WHEN NOT MATCHED THEN
  INSERT (visit_id, patient_name, city, department, consultation_fee, tests_count)
  VALUES (source.visit_id, source.patient_name, source.city,
          source.department, source.consultation_fee, source.tests_count)
""")

spark.sql("SELECT * FROM hospital_visits_target ORDER BY visit_id").show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            6000|          3|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
|     112|Nikhil Joshi|   Nagpur|  Neurology|            7000|          2|
|     113|Pooja Sharma|    Surat|Orthopedics|            3200|          1|
+--------+------------+--

In [0]:
spark.sql("SHOW CATALOGS").show()

+--------------------+
|             catalog|
+--------------------+
|          assessment|
|              bronze|
|               datas|
|hexa_ws_740560962...|
| hexacatalog_student|
|    hospital_catalog|
|     hospitalcatalog|
|           medallion|
|             samples|
|      sensor_catalog|
|              system|
|              task_1|
+--------------------+



Part 8 — Unity Catalog


In [0]:
spark.sql("USE CATALOG hospital_catalog")
spark.sql("USE SCHEMA default")

DataFrame[]

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS hospital_catalog.default.visits (
  visit_id         INT,
  patient_name     STRING,
  city             STRING,
  department       STRING,
  consultation_fee INT,
  tests_count      INT
)
USING DELTA
COMMENT 'Managed hospital visits table in Unity Catalog'
""")

DataFrame[]

In [0]:
base_df.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("hospital_catalog.default.visits")

spark.sql("SELECT * FROM hospital_catalog.default.visits").show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



In [0]:
spark.sql("SHOW TABLES IN hospital_catalog.default").show()

+--------+-------------------+-----------+
|database|          tableName|isTemporary|
+--------+-------------------+-----------+
| default|             visits|      false|
|        |      daily_updates|       true|
|        |    hospital_visits|       true|
|        |new_hospital_visits|       true|
+--------+-------------------+-----------+



PART 9 — DATA GOVERNANCE

In [0]:
spark.sql("SHOW CATALOGS").show()
spark.sql("SHOW SCHEMAS IN hospital_catalog").show()
spark.sql("SHOW TABLES IN hospital_catalog.default").show()

+--------------------+
|             catalog|
+--------------------+
|          assessment|
|              bronze|
|               datas|
|hexa_ws_740560962...|
| hexacatalog_student|
|    hospital_catalog|
|     hospitalcatalog|
|           medallion|
|             samples|
|      sensor_catalog|
|              system|
|              task_1|
+--------------------+

+------------------+
|      databaseName|
+------------------+
|          clinical|
|           default|
|information_schema|
+------------------+

+--------+-------------------+-----------+
|database|          tableName|isTemporary|
+--------+-------------------+-----------+
| default|             visits|      false|
|        |      daily_updates|       true|
|        |    hospital_visits|       true|
|        |new_hospital_visits|       true|
+--------+-------------------+-----------+



In [0]:
spark.sql("""
CREATE OR REPLACE TABLE hospital_catalog.default.high_value_visits
USING DELTA
AS
SELECT * FROM hospital_catalog.default.visits
WHERE consultation_fee >= 5000
""")

spark.sql("SELECT * FROM hospital_catalog.default.high_value_visits").show()

+--------+------------+---------+----------+----------------+-----------+
|visit_id|patient_name|     city|department|consultation_fee|tests_count|
+--------+------------+---------+----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|          1|
|     104|  Priya Nair|Bangalore|Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai| Neurology|            7000|          1|
|     107| Karan Patel|Ahmedabad|Cardiology|            5000|          1|
+--------+------------+---------+----------+----------------+-----------+



In [0]:
spark.sql("DESCRIBE EXTENDED hospital_catalog.default.high_value_visits").show(truncate=False)


+----------------------------+-----------------------------------------------------------------------+-------+
|col_name                    |data_type                                                              |comment|
+----------------------------+-----------------------------------------------------------------------+-------+
|visit_id                    |bigint                                                                 |NULL   |
|patient_name                |string                                                                 |NULL   |
|city                        |string                                                                 |NULL   |
|department                  |string                                                                 |NULL   |
|consultation_fee            |bigint                                                                 |NULL   |
|tests_count                 |bigint                                                                 |NULL   |
|

In [0]:
spark.sql("""
GRANT SELECT ON TABLE hospital_catalog.default.visits
TO `account users`
""")

spark.sql("SHOW GRANTS ON TABLE hospital_catalog.default.visits").show()

+-------------+----------+----------+--------------------+
|    Principal|ActionType|ObjectType|           ObjectKey|
+-------------+----------+----------+--------------------+
|account users|    SELECT|     TABLE|hospital_catalog....|
+-------------+----------+----------+--------------------+



In [0]:
spark.sql("DESCRIBE HISTORY hospital_catalog.default.visits").show(truncate=False)

+-------+-------------------+---------------+-------------------------------------------------------+---------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------+
|version|timestamp          |userId         |userNam

In [0]:
spark.sql("DESCRIBE HISTORY hospital_catalog.default.high_value_visits").show(truncate=False)

+-------+-------------------+---------------+-------------------------------------------------------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------+
|version|timestamp          |userId         |userName                                               |operation                        |operationParameters                                                                                                                                                                             |job |notebook        

In [0]:
spark.sql("SELECT current_user()").show()
spark.sql("SHOW GRANTS ON TABLE hospital_catalog.default.visits").show()

+--------------------+
|      current_user()|
+--------------------+
|azuser5814_mml.lo...|
+--------------------+

+-------------+----------+----------+--------------------+
|    Principal|ActionType|ObjectType|           ObjectKey|
+-------------+----------+----------+--------------------+
|account users|    SELECT|     TABLE|hospital_catalog....|
+-------------+----------+----------+--------------------+



Final Capstone

In [0]:
spark.sql("DROP TABLE IF EXISTS hospital_catalog.default.bronze_raw")

base_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("hospital_catalog.default.bronze_raw")

spark.sql("SELECT * FROM hospital_catalog.default.bronze_raw").show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



In [0]:
from pyspark.sql.functions import col, when

silver = spark.read.table("hospital_catalog.default.bronze_raw") \
    .filter(col("consultation_fee") > 0) \
    .withColumn("total_bill",
        col("consultation_fee") + col("tests_count") * 500) \
    .withColumn("patient_category",
        when(col("consultation_fee") >= 5000, "High")
       .when(col("consultation_fee") >= 3000, "Medium")
       .otherwise("Low"))

spark.sql("DROP TABLE IF EXISTS hospital_catalog.default.silver_clean")
silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("hospital_catalog.default.silver_clean")

spark.sql("SELECT * FROM hospital_catalog.default.silver_clean").show()

+--------+------------+---------+-----------+----------------+-----------+----------+----------------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total_bill|patient_category|
+--------+------------+---------+-----------+----------------+-----------+----------+----------------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|      5500|            High|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|      4000|          Medium|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|      2000|             Low|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      6000|            High|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|      7500|            High|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|      4500|          Medium|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1

In [0]:
from pyspark.sql.functions import count, sum

gold = spark.read.table("hospital_catalog.default.silver_clean") \
    .groupBy("department", "city") \
    .agg(
        count("visit_id").alias("patient_count"),
        sum("consultation_fee").alias("total_revenue"),
        sum("total_bill").alias("total_billing")
    )

spark.sql("DROP TABLE IF EXISTS hospital_catalog.default.gold_analytics")
gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("hospital_catalog.default.gold_analytics")

spark.sql("SELECT * FROM hospital_catalog.default.gold_analytics ORDER BY total_revenue DESC").show()

+-----------+---------+-------------+-------------+-------------+
| department|     city|patient_count|total_revenue|total_billing|
+-----------+---------+-------------+-------------+-------------+
|  Neurology|  Chennai|            1|         7000|         7500|
| Cardiology|Ahmedabad|            1|         5000|         5500|
| Cardiology|Bangalore|            1|         5000|         6000|
| Cardiology|Hyderabad|            1|         5000|         5500|
|Orthopedics|  Kolkata|            1|         3000|         4500|
|Orthopedics|    Delhi|            1|         3000|         4000|
|Dermatology|   Mumbai|            1|         1500|         2000|
|Dermatology|Bangalore|            1|         1500|         2500|
+-----------+---------+-------------+-------------+-------------+



In [0]:
daily = [
    (104, "Priya Nair",  "Bangalore", "Cardiology", 6000, 3),
    (114, "Amit Verma",  "Delhi",     "Neurology",  7500, 2)
]
columns = ["visit_id","patient_name","city","department","consultation_fee","tests_count"]
daily_df = spark.createDataFrame(daily, columns)
daily_df.createOrReplaceTempView("capstone_daily_updates")

spark.sql("""
MERGE INTO hospital_catalog.default.bronze_raw AS t
USING capstone_daily_updates AS s
ON t.visit_id = s.visit_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

spark.sql("SELECT * FROM hospital_catalog.default.bronze_raw ORDER BY visit_id").show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            6000|          3|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
|     114|  Amit Verma|    Delhi|  Neurology|            7500|          2|
+--------+------------+---------+-----------+----------------+-----------+



In [0]:
spark.sql("""
GRANT SELECT ON TABLE hospital_catalog.default.gold_analytics
TO `account users`
""")

spark.sql("DESCRIBE HISTORY hospital_catalog.default.silver_clean").show(truncate=False)
spark.sql("SELECT * FROM hospital_catalog.default.gold_analytics ORDER BY total_revenue DESC").show()

+-------+-------------------+---------------+-------------------------------------------------------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------+
|version|timestamp          |userId         |userName                                               |operation                        |operationParameters                                                                                                                                                                             |job |notebook        